In [3]:
from pathlib import Path

from datasets import load_dataset


def resolve_project_path(relative_path):
    for directory in [Path.cwd(), *Path.cwd().parents]:
        candidate = directory / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


template_path = resolve_project_path("data/templates/sample.raw.10k.templates.parquet")
embedding_path = resolve_project_path("data/embeddings/templates.jina-embeddings-v5-text-small.parquet")

sampled_dataset = load_dataset("parquet", data_files=str(template_path), split="train")
embedding_dataset = load_dataset("parquet", data_files=str(embedding_path), split="train")

if len(sampled_dataset) != len(embedding_dataset):
    raise ValueError(
        f"Row count mismatch: sampled={len(sampled_dataset)}, embeddings={len(embedding_dataset)}"
    )

embedding_column = "embedding_doc_embedding"
dataset = sampled_dataset.add_column(
    embedding_column,
    embedding_dataset[embedding_column],
)

dataset

Dataset({
    features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author', 'embedding_doc', 'embedding_doc_embedding'],
    num_rows: 10000
})

In [4]:
import os

import numpy as np
from sentence_transformers import SentenceTransformer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

model_name = "jinaai/jina-embeddings-v5-text-small"
model = SentenceTransformer(model_name, trust_remote_code=True)
_ = model.encode(["warmup"], task="retrieval", show_progress_bar=False)

import faiss

embedding_matrix = np.asarray(dataset[embedding_column], dtype="float32")
faiss.normalize_L2(embedding_matrix)

embedding_dim = embedding_matrix.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(embedding_matrix)

print(f"FAISS index vectors: {index.ntotal}")
print(f"FAISS index dimension: {index.d}")

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 73262.95it/s]
Default prompt name is set to 'document'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


FAISS index vectors: 10000
FAISS index dimension: 1024


In [6]:
def find_similar_products(query, top_k=10, include_embeddings=False):
    query_embedding = model.encode(
        [query],
        task="retrieval",
        show_progress_bar=False,
    ).astype("float32")
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)
    results = dataset.select(indices[0].tolist()).to_pandas()
    results.insert(0, "similarity_score", scores[0])
    results.insert(0, "rank", range(1, len(results) + 1))

    if not include_embeddings:
        results = results.drop(columns=[embedding_column])

    return results


find_similar_products("blue eye shadow", top_k=5)[
    ["rank", "similarity_score", "title", "embedding_doc", "price"]
]

,rank,similarity_score,title,embedding_doc,price
0,1,0.720755,"Ulta Shimmer Eye Shadow, Mosaic","Product: Ulta Shimmer Eye Shadow, Mosaic\n ...",None
1,2,0.717324,Makeup Geek Foiled Eyeshadow (Blacklight),Product: Makeup Geek Foiled Eyeshadow (Blackli...,None
2,3,0.716008,Shimmering Shades of Soft Shadows That Glide o...,Product: Shimmering Shades of Soft Shadows Tha...,None
3,4,0.699386,"Bella Mari Natural Mineral Eyeshadow, Blue (Pe...","Product: Bella Mari Natural Mineral Eyeshadow,...",None
4,5,0.697439,NESA Eyeshadow Smoky Palette 35 Colors Eye Sha...,Product: NESA Eyeshadow Smoky Palette 35 Color...,None
